## Quick notebook workflow: subset -> featurize -> k-fold CV

`build_dataset` / `build_model` (`soamp.data.factory` / `soamp.model.factory`) are the same two functions `pipeline/train.py` calls for the real run -- here they're driven directly on an in-memory row subset instead of the precomputed artifacts, so featurization and the model always match whatever rows you hand them. Continues from `explore_mic_dataset.ipynb` / `generate_leiden_train_folds.ipynb`.

In [1]:
import numpy as np
import pandas as pd
import torch
import wandb
from dotenv import load_dotenv
from torch.utils.data import DataLoader

from soamp.data.factory import build_dataset
from soamp.data.torch_dataset import LABEL_TO_INT
from soamp.engine.class_balancing import resolve_pos_weight
from soamp.engine.metrics import compute_binary_metrics, logits_to_predictions
from soamp.engine.tracking import build_tracker, watch_model
from soamp.engine.trainer import Trainer
from soamp.model.factory import build_model

load_dotenv()  # picks up WANDB_API_KEY from .env, per .env.example

data_folder_path = "/Users/lukajin/PycharmProjects/soamp/data/"
thresholds_csv_path = "/Users/lukajin/PycharmProjects/soamp/config/thresholds/organism_thresholds.csv"

In [2]:
df = pd.read_csv(data_folder_path + "mic_classification_dataset.csv")
train_df = df[df["split"] != "test"]
train_df["organism"].value_counts()

organism
Escherichia coli          6714
Staphylococcus aureus     5981
Pseudomonas aeruginosa    4348
Name: count, dtype: int64

In [3]:
train_folds = pd.read_csv(data_folder_path + "train_folds_leiden.csv")
fold_columns = ["node_id", "fold_id"]
row_columns = ["peptide_id", "smiles", "organism", "label", "has_noncanonical"]
train_df = pd.merge(
    train_folds[fold_columns], train_df[row_columns],
    left_on="node_id", right_on="peptide_id", how="inner",
)
train_df.head()

,node_id,fold_id,peptide_id,smiles,organism,label,has_noncanonical
0,10,2,10,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,Staphylococcus aureus,active,False
1,11,4,11,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Escherichia coli,active,False
2,11,4,11,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Pseudomonas aeruginosa,active,False
3,11,4,11,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Staphylococcus aureus,active,False
4,15,4,15,C[C@H](NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)[C@H](CO)...,Staphylococcus aureus,inactive,False


### Take an arbitrary subset for a quick experiment

In [4]:
SUBSET_N = 1500
SUBSET_SEED = 42

In [5]:
subset_df = train_df.sample(n=SUBSET_N, random_state=SUBSET_SEED)
subset_df["fold_id"].value_counts()

fold_id
2    323
1    312
0    309
4    278
3    278
Name: count, dtype: int64

### Experiment tracking setup (wandb)

One wandb run for the whole k-fold experiment -- hyperparams/architecture/fold-method are shared across all 5 fold-models, only per-fold metrics vary (logged against `step=fold_id` further down). Requires `wandb login` (or `WANDB_API_KEY` set) once per machine.

In [6]:
EPOCHS = 5
HIDDEN_DIMS = [16, 8]
ORGANISM_EMBED_DIM = 8
SEED = 42
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

# Transcribed by hand from scripts/EDA/generate_leiden_train_folds.ipynb --
# that notebook builds train_folds_leiden.csv and isn't re-run here, so keep
# this in sync manually if its params ever change.
FOLD_GENERATION_METADATA = {
    "method": "sequence-identity graph (blosum45 alignment) + Leiden community "
              "detection; folds = greedy cumulative-count binning of communities "
              "into n_folds groups",
    "identity_threshold": 0.60,
    "substitution_matrix": "blosum45",
    "gap_open": 5,
    "gap_extension": 1,
    "leiden_n_iterations": -1,
    "leiden_seed": 42,
    "n_folds": 5,
    "source_notebook": "scripts/EDA/generate_leiden_train_folds.ipynb",
}

In [7]:
# Not used for training/eval -- built purely to source the real peptide/organism
# representation info (method names, dims) build_tracker logs into the run config.
preview_bundle = build_dataset(row_groups={"fit": subset_df.to_dict("records")})

In [8]:
run = build_tracker(
    preview_bundle,
    hyperparams={
        "epochs": EPOCHS,
        "hidden_dims": HIDDEN_DIMS,
        "organism_embed_dim": ORGANISM_EMBED_DIM,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "subset_n": SUBSET_N,
        "subset_seed": SUBSET_SEED,
        "fold_generation": FOLD_GENERATION_METADATA,
    },
    project="soamp",
    job_type="kfold_cv",
)
dict(run.config)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: ljinc20 (ljinc20-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: setting up run oml70no5


wandb: Tracking run with wandb version 0.28.2


wandb: Run data is saved locally in /Users/lukajin/PycharmProjects/soamp/scripts/EDA/wandb/run-20260906_035607-oml70no5
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run lemon-terrain-21


wandb: ⭐️ View project at https://wandb.ai/ljinc20-free-university-of-tbilisi-/soamp


wandb: 🚀 View run at https://wandb.ai/ljinc20-free-university-of-tbilisi-/soamp/runs/oml70no5


{'epochs': 5,
 'hidden_dims': [16, 8],
 'organism_embed_dim': 8,
 'seed': 42,
 'batch_size': 64,
 'learning_rate': 0.001,
 'subset_n': 1500,
 'subset_seed': 42,
 'fold_generation': {'source_notebook': 'scripts/EDA/generate_leiden_train_folds.ipynb',
  'leiden_seed': 42,
  'gap_extension': 1,
  'n_folds': 5,
  'substitution_matrix': 'blosum45',
  'gap_open': 5,
  'leiden_n_iterations': -1,
  'identity_threshold': 0.6,
  'method': 'sequence-identity graph (blosum45 alignment) + Leiden community detection; folds = greedy cumulative-count binning of communities into n_folds groups'},
 'peptide_method': 'rdkit_descriptors',
 'peptide_feature_dim': 13,
 'descriptor_names': ['MolWt',
  'TPSA',
  'MolLogP',
  'NumHDonors',
  'NumHAcceptors',
  'NumRotatableBonds',
  'FractionCSP3',
  'RingCount',
  'NumAromaticRings',
  'HeavyAtomCount',
  'NumHeteroatoms',
  'LabuteASA',
  'FormalCharge'],
 'organism_method': 'vocab_embedding',
 'organism_output_kind': 'index',
 'organism_vocab_size': 4,
 'or

In [9]:
thresholds_df = pd.read_csv(thresholds_csv_path)
organisms_used = sorted(subset_df["organism"].unique())
organism_thresholds = thresholds_df[
    thresholds_df["match_key"].isin(organisms_used) & (thresholds_df["level"] == "species")
][["match_key", "active_threshold_uM", "inactive_threshold_uM", "source"]]

thresholds_artifact = wandb.Artifact(name="organism_activity_thresholds", type="dataset")
thresholds_artifact.add(wandb.Table(dataframe=organism_thresholds), "organism_thresholds")
run.log_artifact(thresholds_artifact)
organism_thresholds

,match_key,active_threshold_uM,inactive_threshold_uM,source
1,Escherichia coli,32.0,128.0,user-specified 2026-08-15
3,Staphylococcus aureus,32.0,128.0,user-specified 2026-08-15
5,Pseudomonas aeruginosa,32.0,128.0,user-specified 2026-08-15


### One helper: `build_dataset(row_groups=...)` -> `build_model(bundle)` -> short train + eval

`row_groups` takes any caller-chosen keys (`"fit"`/`"val"` for a fold, or `"fit"`/`"eval_other_pop"` below) -- the scaler and organism vocab are fit only on `row_groups["fit"]`, so a held-out fold or population never leaks into featurization.

In [10]:
def train_and_evaluate(row_groups, eval_groups=("fit", "val"), epochs=5, seed=42,
                        hidden_dims=[16, 8], organism_embed_dim=8, batch_size=64,
                        learning_rate=1e-3, watch=False):
    torch.manual_seed(seed)
    bundle = build_dataset(row_groups=row_groups)
    model = build_model(bundle, hidden_dims=hidden_dims, organism_embed_dim=organism_embed_dim)
    if watch:
        # Watches the real model about to be trained -- gradient/parameter
        # histograms and the computation graph populate from its actual
        # forward/backward passes below, not a throwaway preview model.
        watch_model(model)

    fit_loader = DataLoader(bundle.datasets["fit"], batch_size=batch_size, shuffle=True)

    fit_labels = [LABEL_TO_INT[row["label"]] for row in row_groups["fit"]]
    pos_weight = resolve_pos_weight("auto", None, fit_labels)
    loss_fn = torch.nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(pos_weight) if pos_weight is not None else None
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    trainer = Trainer(model, optimizer, loss_fn)

    for _ in range(epochs):
        trainer.train_epoch(fit_loader)

    # Each eval_loader has shuffle=False, so its eval_out arrays line up 1:1
    # with row_groups[group] in order -- safe to reconstruct a per-row table.
    metrics_by_group = {}
    group_frames = []
    for group in eval_groups:
        eval_loader = DataLoader(bundle.datasets[group], batch_size=batch_size)
        eval_out = trainer.evaluate(eval_loader)
        metrics_by_group[group] = compute_binary_metrics(eval_out["logits"], eval_out["labels"])

        probs = 1.0 / (1.0 + np.exp(-eval_out["logits"]))
        group_df = pd.DataFrame(row_groups[group]).reset_index(drop=True)
        group_df["true_label_int"] = eval_out["labels"].astype(int)
        group_df["pred_label_int"] = logits_to_predictions(eval_out["logits"])
        group_df["logit"] = eval_out["logits"]
        group_df["prob"] = probs
        group_df["correct"] = group_df["true_label_int"] == group_df["pred_label_int"]
        group_df["eval_group"] = group
        group_frames.append(group_df)

    results_df = pd.concat(group_frames, ignore_index=True)
    return metrics_by_group, results_df

### K-fold CV over the Leiden clusters

In [11]:
fold_results = []
fold_metrics_rows = []
for fold_id in sorted(subset_df["fold_id"].unique()):
    row_groups = {
        "fit": subset_df[subset_df["fold_id"] != fold_id].to_dict("records"),
        "val": subset_df[subset_df["fold_id"] == fold_id].to_dict("records"),
    }
    metrics_by_group, results_df = train_and_evaluate(
        row_groups, eval_groups=("fit", "val"),
        epochs=EPOCHS, seed=SEED, hidden_dims=HIDDEN_DIMS, organism_embed_dim=ORGANISM_EMBED_DIM,
        batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, watch=True,
    )
    results_df["val_fold_id"] = fold_id
    fold_results.append(results_df)
    for eval_group, m in metrics_by_group.items():
        fold_metrics_rows.append({"val_fold_id": fold_id, "eval_group": eval_group, **m})

    fit_m, val_m = metrics_by_group["fit"], metrics_by_group["val"]
    print(f"fold {fold_id}: "
          f"fit(accuracy={fit_m['accuracy']:.3f}, f1={fit_m['f1']:.3f}, auroc={fit_m['auroc']:.3f}) | "
          f"val(accuracy={val_m['accuracy']:.3f}, f1={val_m['f1']:.3f}, auroc={val_m['auroc']:.3f}) "
          f"(fit={len(row_groups['fit'])}, val={len(row_groups['val'])})")

    wandb.log({
        "fold": fold_id,
        "fit/accuracy": fit_m["accuracy"], "fit/f1": fit_m["f1"], "fit/auroc": fit_m["auroc"],
        "val/accuracy": val_m["accuracy"], "val/f1": val_m["f1"], "val/auroc": val_m["auroc"],
        "n_fit": len(row_groups["fit"]), "n_val": len(row_groups["val"]),
    }, step=fold_id)

cv_results_df = pd.concat(fold_results, ignore_index=True)
fold_metrics_df = pd.DataFrame(fold_metrics_rows)

print("\nmean +/- std across folds:")
summary_df = fold_metrics_df.groupby("eval_group")[["accuracy", "f1", "auroc"]].agg(["mean", "std"])
print(summary_df)

for eval_group in summary_df.index:
    for metric in ["accuracy", "f1", "auroc"]:
        run.summary[f"{eval_group}_{metric}_mean"] = summary_df.loc[eval_group, (metric, "mean")]
        run.summary[f"{eval_group}_{metric}_std"] = summary_df.loc[eval_group, (metric, "std")]

results_artifact = wandb.Artifact(name="cv_results", type="results")
results_artifact.add(wandb.Table(dataframe=cv_results_df), "cv_results")
logged_results_artifact = run.log_artifact(results_artifact)
logged_results_artifact.wait()  # block until server-side commit, so the round-trip
                                 # fetch right below doesn't race the async upload

cv_results_df.head()

wandb: logging graph, to disable use `wandb.watch(log_graph=False)`


fold 0: fit(accuracy=0.667, f1=0.776, auroc=0.734) | val(accuracy=0.696, f1=0.793, auroc=0.767) (fit=1191, val=309)


wandb: logging graph, to disable use `wandb.watch(log_graph=False)`


fold 1: fit(accuracy=0.641, f1=0.754, auroc=0.745) | val(accuracy=0.670, f1=0.771, auroc=0.757) (fit=1188, val=312)


wandb: logging graph, to disable use `wandb.watch(log_graph=False)`


...

wandb: logging graph, to disable use `wandb.watch(log_graph=False)`


fold 3: fit(accuracy=0.634, f1=0.746, auroc=0.745) | val(accuracy=0.629, f1=0.738, auroc=0.752) (fit=1222, val=278)


wandb: logging graph, to disable use `wandb.watch(log_graph=False)`


fold 4: fit(accuracy=0.645, f1=0.750, auroc=0.768) | val(accuracy=0.565, f1=0.704, auroc=0.585) (fit=1222, val=278)

mean +/- std across folds:
            accuracy                  f1               auroc          
                mean       std      mean       std      mean       std
eval_group                                                            
fit         0.641131  0.017501  0.751086  0.016677  0.747725  0.012363
val         0.632106  0.052482  0.744915  0.036577  0.720883  0.076406


,node_id,fold_id,peptide_id,smiles,organism,label,has_noncanonical,true_label_int,pred_label_int,logit,prob,correct,eval_group,val_fold_id
0,3691,1,3691,CC[C@H](C)[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@H]...,Escherichia coli,active,False,1,1,0.753798,0.680006,True,fit,0
1,5272,4,5272,CC[C@H](C)[C@H](NC(=O)[C@H](CCCCN)NC(=O)[C@H](...,Pseudomonas aeruginosa,active,False,1,1,0.204299,0.550898,True,fit,0
2,4173,1,4173,CC[C@H](C)[C@H](NC(=O)[C@H](CC(=O)O)NC(=O)[C@H...,Pseudomonas aeruginosa,active,False,1,1,0.581658,0.641449,True,fit,0
3,3262,1,3262,CC[C@@H](C)[C@@H](NC(=O)[C@@H](CC(C)C)NC(=O)[C...,Escherichia coli,active,True,1,0,-0.297730,0.426112,False,fit,0
4,1451,3,1451,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCC...,Staphylococcus aureus,active,False,1,1,0.350350,0.586703,True,fit,0


In [12]:
cv_results_df[cv_results_df["val_fold_id"] == 0]["fold_id"].value_counts()

fold_id
2    323
1    312
0    309
4    278
3    278
Name: count, dtype: int64

In [13]:
cv_results_df.shape

(7500, 14)

In [14]:
(cv_results_df['fold_id'] == cv_results_df['val_fold_id']).mean()

np.float64(0.2)

### Verify the wandb round-trip, then close the run

Confirms `cv_results` is actually reloadable in a future session (the stated reason for logging it as an Artifact, not just a transient run-logged table) before finishing this run -- everything above this point is the k-fold CV experiment; the cells below are a separate, unlogged scenario.

In [15]:
api = wandb.Api()
reloaded_artifact = api.artifact(f"{run.entity}/{run.project}/cv_results:latest")
reloaded_table = reloaded_artifact.get("cv_results")
reloaded_df = pd.DataFrame(reloaded_table.data, columns=reloaded_table.columns)

assert reloaded_df.shape == cv_results_df.shape, (reloaded_df.shape, cv_results_df.shape)
print(f"round-trip OK: reloaded {reloaded_df.shape} matches cv_results_df {cv_results_df.shape}")

run.finish()

wandb:   1 of 1 files downloaded.  


wandb: updating run metadata


round-trip OK: reloaded (7500, 14) matches cv_results_df (7500, 14)


wandb: uploading media/graph/graph_0_summary_0eb7e401fa87fb32b86e.graph.json; uploading wandb-summary.json


wandb: uploading wandb-summary.json; uploading config.yaml


wandb: uploading history steps 0-4, summary


wandb: 
wandb: Run history:
wandb: fit/accuracy █▄▁▃▅
wandb:    fit/auroc ▁▃▄▃█
wandb:       fit/f1 █▅▁▃▄
wandb:         fold ▁▃▅▆█
wandb:        n_fit ▃▃▁██
wandb:        n_val ▆▆█▁▁
wandb: val/accuracy █▇▃▄▁
wandb:    val/auroc ██▇▇▁
wandb:       val/f1 █▆▂▄▁
wandb: 
wandb: Run summary:
wandb:      fit/accuracy 0.64484
wandb:         fit/auroc 0.76794
wandb:            fit/f1 0.75
wandb: fit_accuracy_mean 0.64113
wandb:  fit_accuracy_std 0.0175
wandb:    fit_auroc_mean 0.74772
wandb:     fit_auroc_std 0.01236
wandb:       fit_f1_mean 0.75109
wandb:        fit_f1_std 0.01668
wandb:              fold 4
wandb:               +12 ...
wandb: 


wandb: 🚀 View run lemon-terrain-21 at: https://wandb.ai/ljinc20-free-university-of-tbilisi-/soamp/runs/oml70no5
wandb: ⭐️ View project at: https://wandb.ai/ljinc20-free-university-of-tbilisi-/soamp
wandb: Synced 4 W&B file(s), 5 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260906_035607-oml70no5/logs


### Different scenario: fit on some organisms, evaluate on a population never seen in training

In [16]:
row_groups = {
    "fit": subset_df[subset_df["organism"] != "Pseudomonas aeruginosa"].to_dict("records"),
    "eval_other_pop": subset_df[subset_df["organism"] == "Pseudomonas aeruginosa"].to_dict("records"),
}
metrics_by_group, other_pop_results_df = train_and_evaluate(row_groups, eval_groups=("fit", "eval_other_pop"))
fit_m, pop_m = metrics_by_group["fit"], metrics_by_group["eval_other_pop"]
print(f"trained on E. coli/S. aureus:")
print(f"  fit(accuracy={fit_m['accuracy']:.3f}, f1={fit_m['f1']:.3f}, auroc={fit_m['auroc']:.3f})")
print(f"  eval_other_pop(accuracy={pop_m['accuracy']:.3f}, f1={pop_m['f1']:.3f}, auroc={pop_m['auroc']:.3f})")

trained on E. coli/S. aureus:
  fit(accuracy=0.652, f1=0.763, auroc=0.759)
  eval_other_pop(accuracy=0.247, f1=0.220, auroc=0.862)


### Analyzing `cv_results_df` by organism / non-canonical status

`cv_results_df` (built in the k-fold cell above) has one row per
`(subset row, cv_fold_id, eval_group)` triple: each subset row is evaluated
by every fold's model -- once as `eval_group="val"` (the one fold that held
its Leiden cluster out) and once as `eval_group="fit"` for every other fold
(where it was in-sample training data). That means K rows per original
subset row for K folds, tagged with `organism`/`has_noncanonical`/`label`
etc. alongside this run's prediction (`prob`, `pred_label_int`, `correct`).

**Always group by `eval_group` too** when slicing -- mixing `"fit"`
(in-sample, memorized) and `"val"` (held-out, generalization) rows into one
average would conflate overfitting with real performance.

In [17]:
print(cv_results_df.groupby(["eval_group", "organism"])["correct"].mean())
print()
print(cv_results_df.groupby(["eval_group", "has_noncanonical"])["correct"].mean())
print()
cv_results_df.loc[
    (cv_results_df["eval_group"] == "val")
    & cv_results_df["has_noncanonical"]
    & (cv_results_df["organism"] == "Escherichia coli")
]

eval_group  organism              
fit         Escherichia coli          0.580775
            Pseudomonas aeruginosa    0.804598
            Staphylococcus aureus     0.604597
val         Escherichia coli          0.568659
            Pseudomonas aeruginosa    0.787356
            Staphylococcus aureus     0.607880
Name: correct, dtype: float64

eval_group  has_noncanonical
fit         False               0.6640
            True                0.5270
val         False               0.6584
            True                0.5080
Name: correct, dtype: float64



,node_id,fold_id,peptide_id,smiles,organism,label,has_noncanonical,true_label_int,pred_label_int,logit,prob,correct,eval_group,val_fold_id
1211,2519,0,2519,CC[C@H](C)[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H...,Escherichia coli,active,True,1,1,0.162354,0.540500,True,val,0
1232,5294,0,5294,C[C@@H]1NC(=O)[C@H](CCCCN)NC(=O)[C@H](C)NC(=O)...,Escherichia coli,inactive,True,0,0,-0.329045,0.418473,True,val,0
1245,5570,0,5570,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](CC...,Escherichia coli,active,True,1,0,-0.369376,0.408692,False,val,0
1287,7298,0,7298,CCCCCCCCCCCC(=O)N[C@@H](CCCN)C(N)=O,Escherichia coli,inactive,True,0,0,-0.771955,0.316056,True,val,0
1314,7465,0,7465,CC[C@H](C)[C@H](N)C(=O)N[C@@H](CCCN)C(=O)N[C@@...,Escherichia coli,active,True,1,0,-0.243826,0.439344,False,val,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7463,5089,4,5089,NCCC[C@H](NC(=O)[C@@H](N)CCCN)C(=O)N[C@@H](Cc1...,Escherichia coli,inactive,True,0,0,-0.722335,0.326879,True,val,4
7465,3315,4,3315,CC(C)C[C@@H]1NC(=O)[C@H](CCCN)NC(=O)[C@H](C(C)...,Escherichia coli,active,True,1,0,-0.563077,0.362836,False,val,4
7467,1713,4,1713,CC(C)C[C@@H]1NC(=O)[C@H](CCCCN)NC(=O)[C@H](C(C...,Escherichia coli,active,True,1,0,-0.558926,0.363796,False,val,4
7484,1688,4,1688,CC(C)C[C@@H]1NC(=O)[C@H](CCCN)NC(=O)[C@H](C(C)...,Escherichia coli,active,True,1,0,-0.680102,0.336239,False,val,4


`pipeline/train.py` calls these same two functions, `build_dataset` and `build_model` -- just with `row_groups=None` (the default), which loads the precomputed artifacts (`peptide_features.csv`, `organism_vocab.json`, `peptide_feature_scaler.json`, `val_split.json`) instead of an in-memory subset. Same code path either way, so the notebook and the real run can never drift apart.